In [3]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/Ayustar/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates("content_id")
print("n =", len(df))

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 143 (delta 53), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.86 MiB | 13.59 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/flyrank-ml-internship-starter
n = 30000


# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayustar/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Signal 1: staleness (flag-linked): CONFIRMED. Pages not updated in 90+ days decline more (about 61% vs 51%, n = 9,345 and 20,655). The 30-89d and 180d+ buckets have under 200 pages each, so I don’t read them. Observational only.

Signal 2: volume: write the printed verdict here (CONFIRMED, OPPOSITE, MIXED, or FALSE) plus one sentence on how the decline rate moves across the four buckets, with n.

Rule: Review first the pages that are stale (90+ days since update) and still visible (500+ impressions in 90 days). Staleness is the risk. Impressions are the stake, since they show how much visibility a page could lose. Score = half staleness, half log impressions.
Reason code: stale_visible_page · Action: refresh (everything else: no_flag / monitor)

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

df["is_down"] = (df["trend_direction"] == "down").astype(int)   # outcome check only, never in the score
base = df["is_down"].mean()
print("overall decline rate:", round(base, 3), " n =", len(df))

# Signal 1: staleness (flag-linked)
df["stale_bucket"] = pd.cut(df["days_since_last_update"], [0, 30, 90, 180, 10_000],
                            labels=["0-29d", "30-89d", "90-179d", "180d+"], right=False)
t1 = df.groupby("stale_bucket", observed=True)["is_down"].agg(n="count", decline_rate="mean").round(3)
print("\nSIGNAL 1: staleness\n", t1)

# Signal 2: volume
df["imp_bucket"] = pd.cut(df["impressions_90d"], [0, 100, 500, 2000, 10_000_000],
                          labels=["1-99", "100-499", "500-1999", "2000+"], right=False)
t2 = df.groupby("imp_bucket", observed=True)["is_down"].agg(n="count", decline_rate="mean").round(3)
print("\nSIGNAL 2: volume\n", t2)

# automatic verdict (ignores buckets with n < 200)
def verdict(t):
    r = t[t["n"] >= 200]["decline_rate"].values
    if r.max() - r.min() < 0.03: return "FALSE"
    if all(np.diff(r) > 0): return "CONFIRMED"
    if all(np.diff(r) < 0): return "OPPOSITE"
    return "MIXED"

print("\nSignal 1 verdict (staleness):", verdict(t1))
print("Signal 2 verdict (volume):", verdict(t2))


overall decline rate: 0.542  n = 30000

SIGNAL 1: staleness
                   n  decline_rate
stale_bucket                     
0-29d         20480         0.511
30-89d          175         0.589
90-179d        9171         0.611
180d+           174         0.471

SIGNAL 2: volume
                 n  decline_rate
imp_bucket                     
1-99         7994         0.389
100-499      5280         0.604
500-1999     6511         0.618
2000+       10215         0.581

Signal 1 verdict (staleness): CONFIRMED
Signal 2 verdict (volume): MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
eligible = (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 500)
stale_part = (df["days_since_last_update"] / 180).clip(upper=1)
vis_part = np.log1p(df["impressions_90d"]) / np.log1p(df["impressions_90d"].max())

df["score"] = (100 * (0.5 * stale_part + 0.5 * vis_part)).round(2)
df.loc[~eligible, "score"] = 0
df["reason_code"] = np.where(eligible, "stale_visible_page", "no_flag")
df["action"] = np.where(eligible, "refresh", "monitor")

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

out_cols = ["rank", "content_id", "score", "reason_code", "action",
            "days_since_last_update", "impressions_90d"]
os.makedirs("work/outputs", exist_ok=True)
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("flagged:", int(eligible.sum()), "of", len(df))

flagged: 6575 of 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)
for _, r in top10.iterrows():
    wrong = []
    if r["avg_position"] <= 10:
        wrong.append("it already ranks on page one, so staleness may be harmless")
    elif r["avg_position"] > 20:
        wrong.append("it ranks too low for a refresh to move it")
    if r["sessions_90d"] < 10:
        wrong.append("few visits, so the demand may be impressions only")
    wrong.append("a related page absorbed its demand, or it is a seasonal dip")
    print(f"{int(r['rank'])}. {r['action']} | stale {int(r['days_since_last_update'])}d, "
          f"{int(r['impressions_90d'])} impressions, score {r['score']}. "
          f"Wrong if: {'; '.join(wrong)}.")

1. refresh | stale 194d, 61678 impressions, score 91.92. Wrong if: a related page absorbed its demand, or it is a seasonal dip.
2. refresh | stale 194d, 59472 impressions, score 91.78. Wrong if: it ranks too low for a refresh to move it; a related page absorbed its demand, or it is a seasonal dip.
3. refresh | stale 194d, 25715 impressions, score 88.59. Wrong if: it ranks too low for a refresh to move it; a related page absorbed its demand, or it is a seasonal dip.
4. refresh | stale 193d, 13299 impressions, score 86.08. Wrong if: a related page absorbed its demand, or it is a seasonal dip.
5. refresh | stale 194d, 7812 impressions, score 84.06. Wrong if: it ranks too low for a refresh to move it; few visits, so the demand may be impressions only; a related page absorbed its demand, or it is a seasonal dip.
6. refresh | stale 193d, 7558 impressions, score 83.94. Wrong if: a related page absorbed its demand, or it is a seasonal dip.
7. refresh | stale 194d, 4590 impressions, score 82.04

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Leakage check: The score uses only days_since_last_update and impressions_90d, both known before the decision point. trend_direction and is_down are used only to check signals and evaluate the queue, never as inputs. No product flags and no future-window data. Weak picks: (write what the weak-picks table shows, then compare the decline rates above to the 54.2% base).

In [12]:
# weak picks in the top 50
t50 = queue.head(50)
weak = t50[(t50["avg_position"] > 20) | (t50["sessions_90d"] < 10)]
print("weak-looking picks in top 50:", len(weak))
print(weak[["rank", "avg_position", "sessions_90d", "impressions_90d"]].head(10))

# honest check against the outcome (evaluation only)
flag = queue[queue["reason_code"] == "stale_visible_page"]
print("\ndecline rate  top 10:", round(queue.head(10)["is_down"].mean(), 3),
      "| top 50:", round(t50["is_down"].mean(), 3),
      "| all flagged:", round(flag["is_down"].mean(), 3),
      "| base:", round(base, 3))

# leakage check
score_inputs = ["days_since_last_update", "impressions_90d"]
print("\nscore inputs:", score_inputs)
print("label-derived inputs in score:", [c for c in score_inputs if c in ("trend_direction", "is_down")])

import json
json.dump({"n": len(df), "base_decline_rate": round(float(base), 3),
           "flagged": int(len(flag)),
           "decline_rate_flagged": round(float(flag["is_down"].mean()), 3)},
          open("work/outputs/baseline_metrics.json", "w"))

weak-looking picks in top 50: 19
    rank  avg_position  sessions_90d  impressions_90d
1      2          24.8            82            59472
2      3          22.2            80            25715
4      5          39.0             5             7812
6      7          31.0             4             4590
8      9          25.3            12             4429
11    12          27.9          4218           443434
12    13          15.8             3             1697
13    14           7.8             9             1408
15    16          21.8             6             1316
19    20          26.2           626           286608

decline rate  top 10: 0.9 | top 50: 0.58 | all flagged: 0.616 | base: 0.542

score inputs: ['days_since_last_update', 'impressions_90d']
label-derived inputs in score: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.